In [1]:
!pip install kagglehub

In [2]:
import kagglehub
path = kagglehub.dataset_download("smayanj/netflix-users-database")
print("Path to dataset files:", path)

100%|██████████| 354k/354k [00:00<00:00, 1.03MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/smayanj/netflix-users-database/versions/1


# **Датасет**
В качестве датасета я выбрал Netflix Userbase Dataset, в котором представлены различные данные об активности пользователей Netflix.

In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from itertools import combinations
import os

file_name = os.listdir(path)[0]
df = pd.read_csv(os.path.join(path, file_name))

print(df.head())
print(df.info())

   User_ID            Name  Age Country Subscription_Type  Watch_Time_Hours  \
0        1  James Martinez   18  France           Premium             80.26   
1        2     John Miller   23     USA           Premium            321.75   
2        3      Emma Davis   60      UK             Basic             35.89   
3        4     Emma Miller   44     USA           Premium            261.56   
4        5      Jane Smith   68     USA          Standard            909.30   

  Favorite_Genre  Last_Login  
0          Drama  2024-05-12  
1         Sci-Fi  2025-02-05  
2         Comedy  2025-01-24  
3    Documentary  2024-03-25  
4          Drama  2025-01-14  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   User_ID            25000 non-null  int64  
 1   Name               25000 non-null  object 
 2   Age                25000 non-null 

# **Тесты Стьюдента, Манна-Уитни, Фишера**
Для начала сравним среднее значение возраста и времени просмотра у пользователей с типами подписки: Basic и Premium.

Как мы видим, все p-value у всех тестов значительно больше 0.05. В случае теста Стьюдента это означает, что различия между средними значениями у пользователей с разными типами подписки статистически незначимо и мы можем принять их примерно равными.
Тест Манна-Уитни подтверждает подтверждает результат t-теста. Нет значимого сдвига в распределениях, типичные пользователи обеих групп ведут себя одинаково.
Тест Фишера показывает, что различия в дисперсиях (разбросе данных) не обнаружены. Это означает, что вовлеченность пользователей однородна в обоих тарифах.

In [20]:
group_p = df[df['Subscription_Type'] == 'Premium']
group_b = df[df['Subscription_Type'] == 'Basic']
group_s = df[df['Subscription_Type'] == 'Standard']

metrics = ['Watch_Time_Hours', 'Age']
results = []

for m in metrics:
    data_a = group_p[m].dropna()
    data_b = group_b[m].dropna()

    t_stat, t_p = stats.ttest_ind(data_a, data_b)

    u_stat, u_p = stats.mannwhitneyu(data_a, data_b)

    f_stat = data_a.var() / data_b.var()
    df1, df2 = len(data_a) - 1, len(data_b) - 1
    f_p = 2 * min(stats.f.cdf(f_stat, df1, df2), 1 - stats.f.cdf(f_stat, df1, df2))

    results.append({
        "Метрика": m,
        "Среднее Premium": round(data_a.mean(), 2),
        "Среднее Basic": round(data_b.mean(), 2),
        "T-test (p-value)": round(t_p, 4),
        "Mann-Whitney (p-value)": round(u_p, 4),
        "Fisher F-test (p-value)": round(f_p, 4)
    })

df_results = pd.DataFrame(results)
display(df_results)

,Метрика,Среднее Premium,Среднее Basic,T-test (p-value),Mann-Whitney (p-value),Fisher F-test (p-value)
0,Watch_Time_Hours,501.41,502.99,0.7203,0.7146,0.6507
1,Age,46.56,46.35,0.4907,0.4887,0.3013


# **Anova (Фишер для нескольких сегментов) и попарные сравнения тестов Стьюдента и Фишера**

Сравним время просмотра для разных типов подписки. Сначала посчитаем Anova для всех трёх сегментов. Как мы видим, p-value значительно больше 0.05, а значит среднее время просмотра примерно одинаково между тремя группами. Это подтверждают и попарные сравнения: везде p-value значительно больше 0.05.
По t-тесту мы видим, что средние значения между группами одинаковы. Тест Фишера дополнительно подтвердил, что поведение пользователей не только схоже в среднем, но и обладает одинаковой степенью вариативности (разброса) во всех трех сегмента. Таким образом, метрика вовлеченности не зависит от выбранного тарифного плана.

In [21]:
f_anova, p_anova = stats.f_oneway(
    df[df['Subscription_Type'] == 'Basic']['Watch_Time_Hours'],
    df[df['Subscription_Type'] == 'Standard']['Watch_Time_Hours'],
    df[df['Subscription_Type'] == 'Premium']['Watch_Time_Hours']
)

print(f"\nФишер для 3х сегментов (ANOVA) по разным типам подписки:")
print(f"по метрике Watch_Time_Hours: p-value = {p_anova:.4f}")

segments = ['Basic', 'Standard', 'Premium']
pairs = list(combinations(segments, 2))
results_3 = []

for s1, s2 in pairs:
    d1 = df[df['Subscription_Type'] == s1]['Watch_Time_Hours']
    d2 = df[df['Subscription_Type'] == s2]['Watch_Time_Hours']

    t_p = stats.ttest_ind(d1, d2).pvalue

    f_stat = d1.var() / d2.var()
    f_p = 2 * min(stats.f.cdf(f_stat, len(d1)-1, len(d2)-1), 1 - stats.f.cdf(f_stat, len(d1)-1, len(d2)-1))

    results_3.append({
        "Пара": f"{s1} vs {s2}",
        "T-test p-value": round(t_p, 4),
        "Fisher p-value": round(f_p, 4),
        "Разница средних": round(d1.mean() - d2.mean(), 2)
    })

print("\nПопарные сравнения Watch_Time_Hours")
display(pd.DataFrame(results_3))


Фишер для 3х сегментов (ANOVA) по разным типам подписки:
по метрике Watch_Time_Hours: p-value = 0.3706

Попарные сравнения Watch_Time_Hours


,Пара,T-test p-value,Fisher p-value,Разница средних
0,Basic vs Standard,0.1733,0.9238,6.05
1,Basic vs Premium,0.7203,0.6507,1.59
2,Standard vs Premium,0.3156,0.7223,-4.46


# **Точный и Эфронов ДИ**

Доверительные интервалы, построенные точным методом и методом Эфрона (бутстрап), практически идентичны, что характерно для выборок большого объема. Высокая степень пересечения интервалов для групп Premium и Basic по всем метрикам полностью соответствует результатам теста Стьюдента: так как интервалы накладываются друг на друга, разница между средними значениями не является статистически значимой.

In [24]:
def get_bootstrap_ci(data, n_bootstrap=2000):
    boot_means = []
    for _ in range(n_bootstrap):
        boot_sample = np.random.choice(data, size=len(data), replace=True)
        boot_means.append(np.mean(boot_sample))
    return np.percentile(boot_means, [2.5, 97.5])

metrics = ['Watch_Time_Hours', 'Age']
ci_results = []

for m in metrics:
    for segment in ['Premium', 'Basic']:
        data = df[df['Subscription_Type'] == segment][m].dropna()
        n = len(data)
        mean = np.mean(data)
        std_err = stats.sem(data)

        exact_ci = stats.t.interval(0.95, df=n-1, loc=mean, scale=std_err)

        boot_ci = get_bootstrap_ci(data)

        ci_results.append({
            "Метрика": m,
            "Сегмент": segment,
            "Среднее": round(mean, 2),
            "Точный ДИ": np.round(exact_ci, 2),
            "Эфронов ДИ": np.round(boot_ci, 2),
            "Ширина Точного": round(exact_ci[1] - exact_ci[0], 3),
            "Ширина Эфронова": round(boot_ci[1] - boot_ci[0], 3)
        })

df_ci = pd.DataFrame(ci_results)
display(df_ci)

,Метрика,Сегмент,Среднее,Точный ДИ,Эфронов ДИ,Ширина Точного,Ширина Эфронова
0,Watch_Time_Hours,Premium,501.41,"[495.27, 507.55]","[495.23, 507.26]",12.285,12.030
1,Watch_Time_Hours,Basic,502.99,"[496.87, 509.12]","[496.76, 509.1]",12.258,12.342
2,Age,Premium,46.56,"[46.14, 46.97]","[46.14, 46.98]",0.832,0.841
3,Age,Basic,46.35,"[45.93, 46.77]","[45.94, 46.79]",0.844,0.846


# **Другие статистические тесты**

Помимо использованных ранее статистических тестов, можно применить тест Хи-квадрат Пирсона для проверки связи между категориальными переменными. Например, проверить, есть ли статистическая зависимость между страной и любимым жанром. Также было бы целесообразно применить тест Шапиро-Уилка для строгой проверки условий применимости теста Стьюдента. Это позволило бы подтвердить, что выбор параметрических методов был математически обоснован.